<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/TC_25a_flash_attention_from_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MASTER-02: FlashAttention — From Memory Bottlenecks to Tiled Computation

**Time**: ~4 hours | **Goal**: Build FlashAttention intuition from scratch

**The problem this solves**: Standard attention creates an N×N matrix that doesn't fit in GPU fast memory (SRAM). FlashAttention computes attention WITHOUT ever materializing the full N×N matrix.

**What you'll build through 8 progressive stages**:
1. Standard attention (understand what we're optimizing)
2. The memory hierarchy (why N×N is a problem)
3. The softmax problem (why you can't just tile naively)
4. Online softmax (the key mathematical trick)
5. Block-by-block attention (tiled computation)
6. Full FlashAttention (putting it all together)
7. Numerical verification
8. Connection to real implementations

**Every stage has exercises with assertions. No skipping.**

In [1]:
import torch
import torch.nn.functional as F
import math
import time

## Stage 1: Standard Attention — What We're Optimizing (~15 min)

In [2]:
# Exercise 1.1: Implement standard scaled dot-product attention

def standard_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                       mask: bool = True) -> torch.Tensor:
    """Standard attention: O = softmax(QK^T / sqrt(d)) @ V

    Args:
        Q, K, V: shape (batch, n_heads, seq_len, head_dim)
        mask: if True, apply causal mask
    Returns:
        output: shape (batch, n_heads, seq_len, head_dim)

    Steps:
    1. Compute scores: S = Q @ K^T / sqrt(d)     → shape (..., N, N)
    2. If mask: fill upper triangle with -inf
    3. Compute weights: P = softmax(S, dim=-1)    → shape (..., N, N)
    4. Compute output: O = P @ V                  → shape (..., N, d)
    """

    # YOUR CODE HERE
    B, num_heads, N, d_head = Q.shape
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_head)
    if mask:
        mask = torch.triu(torch.ones(N, N, dtype=torch.bool))
        scores.masked_fill_(mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output

# Test
torch.manual_seed(42)
B, H, N, D = 1, 1, 8, 4  # small for debugging
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O = standard_attention(Q, K, V, mask=True)
assert O.shape == (B, H, N, D)
print(f'✅ Stage 1.1: Standard attention, output shape {O.shape}')

✅ Stage 1.1: Standard attention, output shape torch.Size([1, 1, 8, 4])


In [3]:
# Exercise 1.2: Count the memory usage

def attention_memory(N: int, D: int, dtype_bytes: int = 2) -> dict:
    """Calculate memory for standard attention (per head, per batch).

    Returns dict with:
    - Q, K, V: N*D each
    - S (score matrix): N*N  ← THE PROBLEM
    - P (softmax output): N*N
    - O (output): N*D
    """
    return {
        'Q': N * D * dtype_bytes,
        'K': N * D * dtype_bytes,
        'V': N * D * dtype_bytes,
        'S_scores': N * N * dtype_bytes,  # ← THIS dominates
        'P_softmax': N * N * dtype_bytes,  # ← THIS too
        'O_output': N * D * dtype_bytes,
    }

# Calculate for realistic sizes
for N in [1024, 4096, 16384, 65536, 131072]:
    mem = attention_memory(N, 128)
    total = sum(mem.values())
    score_mem = mem['S_scores'] + mem['P_softmax']
    print(f'  N={N:>6d}: Total={total/1e6:>8.1f}MB, Score matrices={score_mem/1e6:>8.1f}MB ({score_mem/total*100:.0f}%)')

print(f'\nAt N=131072 (128K context): score matrix alone needs {131072**2*2/1e9:.1f}GB!')
print('GPU SRAM (L1 cache) is only ~20MB. HBM is 80GB but SLOW.')
print('Standard attention is MEMORY BOUND, not compute bound.')

  N=  1024: Total=     5.2MB, Score matrices=     4.2MB (80%)
  N=  4096: Total=    71.3MB, Score matrices=    67.1MB (94%)
  N= 16384: Total=  1090.5MB, Score matrices=  1073.7MB (98%)
  N= 65536: Total= 17247.0MB, Score matrices= 17179.9MB (100%)
  N=131072: Total= 68853.7MB, Score matrices= 68719.5MB (100%)

At N=131072 (128K context): score matrix alone needs 34.4GB!
GPU SRAM (L1 cache) is only ~20MB. HBM is 80GB but SLOW.
Standard attention is MEMORY BOUND, not compute bound.


## Stage 2: The GPU Memory Hierarchy (~15 min)

**Why does memory matter more than compute?**

```
GPU Memory Hierarchy:
┌─────────────────────────────────┐
│  SRAM (on-chip, per SM)          │  ~20MB total, ~19 TB/s
│  This is where computation lives  │  Registers + shared memory
├─────────────────────────────────┤
│  HBM (off-chip, DRAM)            │  80GB, ~3 TB/s
│  This is where tensors live       │  Main GPU memory
└─────────────────────────────────┘
```
SRAM is ~6x faster than HBM. Standard attention reads/writes the N×N matrix to HBM repeatedly. FlashAttention keeps everything in SRAM by processing in small tiles.

In [4]:
# Exercise 2.1: Calculate memory transfer costs

def attention_hbm_transfers(N: int, D: int, dtype_bytes: int = 2) -> dict:
    """How many bytes are read/written to HBM in standard attention?"""
    return {
        'read_Q': N * D * dtype_bytes,
        'read_K': N * D * dtype_bytes,
        'write_S': N * N * dtype_bytes,  # materialize scores in HBM
        'read_S_for_softmax': N * N * dtype_bytes,  # read back for softmax
        'write_P': N * N * dtype_bytes,  # write softmax output
        'read_P_for_matmul': N * N * dtype_bytes,  # read back for P@V
        'read_V': N * D * dtype_bytes,
        'write_O': N * D * dtype_bytes,
    }

N, D = 4096, 128
standard = attention_hbm_transfers(N, D)
total_standard = sum(standard.values())
print(f'Standard attention HBM transfers (N={N}, D={D}):')
for name, bytes_ in standard.items():
    print(f'  {name:25s}: {bytes_/1e6:.1f}MB')
print(f'  {"TOTAL":25s}: {total_standard/1e6:.1f}MB')
print(f'\nFlashAttention eliminates the N×N writes to HBM entirely.')

flash_transfers = (N * D * 2 * 3 + N * D * 2) * 2  # Q,K,V read + O write
print(f'FlashAttention HBM transfers: ~{flash_transfers/1e6:.1f}MB')
print(f'Savings: {total_standard/flash_transfers:.1f}x less HBM traffic')

Standard attention HBM transfers (N=4096, D=128):
  read_Q                   : 1.0MB
  read_K                   : 1.0MB
  write_S                  : 33.6MB
  read_S_for_softmax       : 33.6MB
  write_P                  : 33.6MB
  read_P_for_matmul        : 33.6MB
  read_V                   : 1.0MB
  write_O                  : 1.0MB
  TOTAL                    : 138.4MB

FlashAttention eliminates the N×N writes to HBM entirely.
FlashAttention HBM transfers: ~8.4MB
Savings: 16.5x less HBM traffic


## Stage 3: Why You Can't Just Tile Naively — The Softmax Problem (~20 min)

You might think: just process the N×N matrix in blocks! But softmax needs to see the ENTIRE row to compute the denominator.

```
softmax(x_i) = exp(x_i) / Σ_j exp(x_j)
                          ^^^^^^^^^^^^^^^^
                          need ALL x values!
```
If you process block 1 first, you don't know the full sum yet. When you process block 2, the denominator changes. You'd need to go back and fix block 1.

In [5]:
# Exercise 3.1: Show the problem concretely

scores = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])

# Full softmax (correct)
full_softmax = torch.softmax(scores, dim=0)

# Naive block softmax (WRONG)
block1 = torch.softmax(scores[:3], dim=0)  # softmax of first 3
block2 = torch.softmax(scores[3:], dim=0)  # softmax of last 3
naive_result = torch.cat([block1, block2])

print(f'Full softmax:  {full_softmax}')
print(f'Naive blocks:  {naive_result}')
print(f'Match? {torch.allclose(full_softmax, naive_result)}')
print()
print('They are DIFFERENT because each block used a different denominator.')
print('Block 1 denominator: exp(1)+exp(2)+exp(3)')
print('Block 2 denominator: exp(4)+exp(5)+exp(6)')
print('Correct denominator: exp(1)+exp(2)+...+exp(6)  ← need ALL values')

Full softmax:  tensor([0.0043, 0.0116, 0.0315, 0.0858, 0.2331, 0.6337])
Naive blocks:  tensor([0.0900, 0.2447, 0.6652, 0.0900, 0.2447, 0.6652])
Match? False

They are DIFFERENT because each block used a different denominator.
Block 1 denominator: exp(1)+exp(2)+exp(3)
Block 2 denominator: exp(4)+exp(5)+exp(6)
Correct denominator: exp(1)+exp(2)+...+exp(6)  ← need ALL values


In [6]:
torch.softmax(torch.tensor([1.0, 2.0, 3.0]), dim=0)

tensor([0.0900, 0.2447, 0.6652])

In [7]:
# Exercise 3.2: The numerical stability issue too
# Before we fix the tiling issue, understand the max-subtract trick

# BAD: exp(1000) = inf!
bad_scores = torch.tensor([1000.0, 1001.0, 1002.0])
try:
    result = torch.exp(bad_scores)  # overflows!
    print(f'exp of large values: {result}')
except:
    pass

# GOOD: subtract max first, then exp
# softmax(x) = exp(x - max(x)) / Σ exp(x - max(x))
# Mathematically identical but numerically stable

# YOUR TASK: Implement safe softmax from scratch
def safe_softmax(x: torch.Tensor) -> torch.Tensor:
    """Numerically stable softmax.
    1. Find max value
    2. Subtract max from all values
    3. Compute exp
    4. Divide by sum
    """
    # YOUR CODE HERE
    x_exp = torch.exp(x - x.max(dim=-1)[0])
    return x_exp / x_exp.sum(dim=-1, keepdim=True)

result = safe_softmax(bad_scores)
assert torch.allclose(result, torch.softmax(bad_scores, dim=0))
assert torch.allclose(result.sum(), torch.tensor(1.0))
print(f'Safe softmax of [1000, 1001, 1002]: {result}')
print('✅ Stage 3.2: Safe softmax')

exp of large values: tensor([inf, inf, inf])
Safe softmax of [1000, 1001, 1002]: tensor([0.0900, 0.2447, 0.6652])
✅ Stage 3.2: Safe softmax


## Stage 4: Online Softmax — Process Blocks Without Going Back (~30 min)

**This is the mathematical core of FlashAttention.**

The trick: maintain a RUNNING max and RUNNING sum. When a new block arrives with a larger value, RESCALE the previous results.

```
After block 1: max1, sum1, output1 = softmax of block 1
After block 2:
  new_max = max(max1, max2)
  correction = exp(max1 - new_max)  ← rescale factor for block 1
  new_sum = sum1 * correction + sum_of_block2_with_new_max
  output = output1 * correction * (old_sum/new_sum) + block2_contribution
```
You process each block ONCE. No going back.

In [8]:
# Exercise 4.1: Online max — process values one at a time

def online_max(values: torch.Tensor) -> float:
    """Find max by processing one element at a time.
    This simulates not having access to all values at once."""
    running_max = float('-inf')
    for v in values:
        running_max = max(running_max, v.item())
    return running_max

assert online_max(torch.tensor([3.0, 1.0, 5.0, 2.0])) == 5.0
print('✅ 4.1: Online max (trivial but important base case)')

✅ 4.1: Online max (trivial but important base case)


In [9]:
# Exercise 4.2: Online softmax denominator — the actual trick
import numpy as np

def online_softmax_stats(values: torch.Tensor, block_size: int = 2):
    """Compute softmax statistics (max, sum_exp) by processing blocks.

    After processing all blocks, we should have:
    - global_max = max over all values
    - global_sum = Σ exp(v - global_max) over all values

    These are enough to compute the full softmax for any element:
    softmax(v_i) = exp(v_i - global_max) / global_sum

    The KEY operation is rescaling when a new block has a larger max.
    """
    global_max = float('-inf')
    global_sum_exp = 0.0

    # Process in blocks
    for start in range(0, len(values), block_size):
        block = values[start:start + block_size]
        block_max = block.max().item()

        # YOUR CODE HERE
        # If block_max > global_max:
        #   1. Rescale existing sum: global_sum_exp *= exp(global_max - block_max)
        #   2. Add new block's contribution: + sum(exp(block - block_max))
        #   3. Update global_max = block_max
        # Else:
        #   Just add: global_sum_exp += sum(exp(block - global_max))
        if block_max > global_max:
            correction_factor = np.exp(global_max - block_max)
            global_sum_exp *= correction_factor
            global_max = block_max
        global_sum_exp += sum(torch.exp(block - global_max))

    return global_max, global_sum_exp

# Test: compare with full softmax
values = torch.tensor([1.0, 3.0, 2.0, 5.0, 4.0, 0.5])

online_max, online_sum = online_softmax_stats(values, block_size=2)

# Full computation for reference
full_max = values.max().item()
full_sum = torch.exp(values - full_max).sum().item()

print(f'Online: max={online_max:.4f}, sum_exp={online_sum:.4f}')
print(f'Full:   max={full_max:.4f}, sum_exp={full_sum:.4f}')
assert abs(online_max - full_max) < 1e-6
assert abs(online_sum - full_sum) < 1e-4
print('\n✅ Stage 4.2: Online softmax stats match full computation!')

Online: max=5.0000, sum_exp=1.5824
Full:   max=5.0000, sum_exp=1.5824

✅ Stage 4.2: Online softmax stats match full computation!


In [10]:
# Exercise 4.3: Online softmax with OUTPUT accumulation
# This is the full algorithm — compute softmax AND weighted sum (attention) online

def online_attention_1d(scores: torch.Tensor, values: torch.Tensor,
                        block_size: int = 2) -> torch.Tensor:
    """Compute softmax(scores) @ values by processing blocks.

    Simulates one ROW of the attention matrix.
    scores: shape (N,) — one row of Q@K^T
    values: shape (N, D) — the V matrix

    Returns: shape (D,) — one row of the output

    Algorithm:
    For each block of scores/values:
        1. Compute block_max
        2. If block_max > running_max:
           a. correction = exp(running_max - block_max)
           b. Rescale running_output *= correction
           c. Rescale running_sum *= correction
           d. running_max = block_max
        3. block_exp = exp(block_scores - running_max)
        4. running_sum += sum(block_exp)
        5. running_output += block_exp @ block_values

    Final: output = running_output / running_sum
    """
    N = scores.shape[0]
    D = values.shape[1]

    running_max = float('-inf')
    running_sum = 0.0
    running_output = torch.zeros(D)

    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        block_scores = scores[start:end]
        block_values = values[start:end]

        # YOUR CODE HERE
        block_max = block_scores.max().item()
        if block_max > running_max:
            correction_factor = np.exp(running_max - block_max)
            running_sum *= correction_factor
            running_output *= correction_factor
            running_max = block_max
        block_exp = torch.exp(block_scores - running_max)
        running_sum += torch.sum(block_exp)
        running_output += block_exp @ block_values

    return running_output / running_sum

# Test against standard attention
torch.manual_seed(42)
N, D = 12, 4
scores = torch.randn(N)
values = torch.randn(N, D)

# Standard
weights = torch.softmax(scores, dim=0)
standard_output = weights @ values

# Online (block_size=3)
online_output = online_attention_1d(scores, values, block_size=3)

print(f'Standard: {standard_output}')
print(f'Online:   {online_output}')
assert torch.allclose(standard_output, online_output, atol=1e-5)
print('\n✅ Stage 4.3: Online attention matches standard attention!')
print('This IS FlashAttention in 1D — process blocks, rescale, never go back.')

Standard: tensor([-0.5880,  0.3701, -0.0967,  0.7911])
Online:   tensor([-0.5880,  0.3701, -0.0967,  0.7911])

✅ Stage 4.3: Online attention matches standard attention!
This IS FlashAttention in 1D — process blocks, rescale, never go back.


## Stage 5: Full 2D Blocked Attention (~30 min)

Now extend from 1D (one query row) to 2D (all query rows, blocked).

```
For each block of Q rows (block_q):
  For each block of K/V rows (block_kv):
    1. Compute block scores: S_block = block_q @ block_k^T / sqrt(d)
    2. Update running max, sum, output using online softmax
  Final: normalize output by running_sum
```

In [11]:
def flash_attention_manual(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                            block_size_q: int = 4, block_size_kv: int = 4,
                            causal: bool = True) -> torch.Tensor:
    """FlashAttention: blocked attention with online softmax.

    Args:
        Q, K, V: shape (N, D)  — single head, single batch for clarity
        block_size_q: tile size along query dimension
        block_size_kv: tile size along key/value dimension
        causal: apply causal masking
    Returns:
        O: shape (N, D)
    """
    N, D = Q.shape
    O = torch.zeros(N, D)
    L = torch.zeros(N)  # running sum of exp per query row
    M = torch.full((N,), float('-inf'))  # running max per query row

    scale = 1.0 / math.sqrt(D)

    # Outer loop: blocks of queries
    for q_start in range(0, N, block_size_q):
        q_end = min(q_start + block_size_q, N)
        Q_block = Q[q_start:q_end]  # shape (Bq, D)

        # Inner loop: blocks of keys/values
        for kv_start in range(0, N, block_size_kv):
            kv_end = min(kv_start + block_size_kv, N)
            K_block = K[kv_start:kv_end]  # shape (Bkv, D)
            V_block = V[kv_start:kv_end]

            # 1. Compute block scores
            S_block = (Q_block @ K_block.T) * scale  # (Bq, Bkv)

            # 2. Apply causal mask if needed
            if causal:
                for qi in range(S_block.shape[0]):
                    for ki in range(S_block.shape[1]):
                        if (q_start + qi) < (kv_start + ki):
                            S_block[qi, ki] = float('-inf')

            # 3. Online softmax update
            # YOUR CODE HERE
            # For each query row qi in this block:
            #   a. block_max = max of S_block[qi]
            #   b. If block_max > M[q_start+qi]:
            #        correction = exp(M[q_start+qi] - block_max)
            #        O[q_start+qi] *= correction
            #        L[q_start+qi] *= correction
            #        M[q_start+qi] = block_max
            #   c. block_exp = exp(S_block[qi] - M[q_start+qi])
            #   d. L[q_start+qi] += sum(block_exp)
            #   e. O[q_start+qi] += block_exp @ V_block
            for qi in range(S_block.shape[0]):
                block_max = S_block[qi].max().item()
                if block_max > M[q_start + qi]:
                    correction_factor = np.exp(M[q_start + qi] - block_max)
                    O[q_start + qi] *= correction_factor
                    L[q_start + qi] *= correction_factor
                    M[q_start + qi] = block_max
                block_exp = torch.exp(S_block[qi] - M[q_start + qi])
                L[q_start + qi] += torch.sum(block_exp)
                O[q_start + qi] += block_exp @ V_block

    # Final normalization
    O = O / L.unsqueeze(1)
    return O

In [12]:
# === VERIFICATION ===
torch.manual_seed(42)
N, D = 16, 8
Q = torch.randn(N, D)
K = torch.randn(N, D)
V = torch.randn(N, D)

# Standard attention (for reference)
scale = 1.0 / math.sqrt(D)
S = (Q @ K.T) * scale
mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
S.masked_fill_(mask, float('-inf'))
P = torch.softmax(S, dim=-1)
O_standard = P @ V

# FlashAttention
O_flash = flash_attention_manual(Q, K, V, block_size_q=4, block_size_kv=4, causal=True)

max_diff = (O_standard - O_flash).abs().max().item()
print(f'Max difference: {max_diff:.8f}')
assert max_diff < 1e-4, f'Too large: {max_diff}'

print(f'✅ FlashAttention matches standard attention!')
print(f'   N={N}, D={D}, block_size=4')
print(f'   Max diff: {max_diff:.2e}')

Max difference: 0.00000015
✅ FlashAttention matches standard attention!
   N=16, D=8, block_size=4
   Max diff: 1.49e-07


/tmp/ipykernel_1407/115315265.py:57: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  correction_factor = np.exp(M[q_start + qi] - block_max)


In [13]:
# Exercise 5.2: Test with different block sizes to prove it always works

for bs_q, bs_kv in [(2, 2), (4, 8), (8, 4), (1, 16), (16, 1)]:
    O_flash = flash_attention_manual(Q, K, V, block_size_q=bs_q, block_size_kv=bs_kv, causal=True)
    diff = (O_standard - O_flash).abs().max().item()
    status = '✅' if diff < 1e-4 else '❌'
    print(f'  block_q={bs_q:2d}, block_kv={bs_kv:2d}: max_diff={diff:.2e} {status}')

print('\nAll block sizes produce the same result — the algorithm is correct.')
print('The block size only affects performance (SRAM utilization), not correctness.')

  block_q= 2, block_kv= 2: max_diff=2.38e-07 ✅
  block_q= 4, block_kv= 8: max_diff=1.79e-07 ✅
  block_q= 8, block_kv= 4: max_diff=1.49e-07 ✅
  block_q= 1, block_kv=16: max_diff=1.49e-07 ✅
  block_q=16, block_kv= 1: max_diff=1.79e-07 ✅

All block sizes produce the same result — the algorithm is correct.
The block size only affects performance (SRAM utilization), not correctness.


/tmp/ipykernel_1407/115315265.py:57: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  correction_factor = np.exp(M[q_start + qi] - block_max)


## Stage 6: Why FlashAttention Is Fast — Memory Analysis (~15 min)

In [14]:
# Exercise 6.1: Count memory per block vs standard

def flash_memory_analysis(N: int, D: int, Bq: int, Bkv: int, dtype_bytes: int = 2):
    """Memory used at any point during FlashAttention."""
    return {
        'Q_block': Bq * D * dtype_bytes,     # loaded from HBM
        'K_block': Bkv * D * dtype_bytes,     # loaded from HBM
        'V_block': Bkv * D * dtype_bytes,     # loaded from HBM
        'S_block': Bq * Bkv * dtype_bytes,    # computed in SRAM, never written to HBM!
        'O_accumulator': Bq * D * dtype_bytes, # kept in SRAM
        'L_M_stats': Bq * 2 * dtype_bytes,    # running max and sum
    }

N, D = 4096, 128
Bq, Bkv = 64, 64

standard = attention_memory(N, D)
flash = flash_memory_analysis(N, D, Bq, Bkv)

print(f'Standard attention peak SRAM need: {sum(standard.values())/1e6:.1f}MB')
print(f'  (N×N score matrix alone: {standard["S_scores"]/1e6:.1f}MB)')
print(f'\nFlashAttention peak SRAM need: {sum(flash.values())/1e3:.1f}KB')
print(f'  (Fits in ~20MB SRAM easily!)')
print(f'\nMemory reduction: {sum(standard.values())/sum(flash.values()):.0f}x')

Standard attention peak SRAM need: 71.3MB
  (N×N score matrix alone: 33.6MB)

FlashAttention peak SRAM need: 74.0KB
  (Fits in ~20MB SRAM easily!)

Memory reduction: 964x


## Stage 7: Connection to Real Implementations

| Your code | Real FlashAttention |
|---|---|
| Python loops over blocks | Triton/CUDA with block-level parallelism |
| `S_block = Q_block @ K_block.T` | `tl.dot(Q_block, K_block_T)` in SRAM |
| `correction = exp(old_max - new_max)` | Same math, vectorized |
| Sequential outer/inner loop | GPU grid: each thread block handles one Q block |
| O(N²) HBM transfers | O(N²D²/SRAM) — proven optimal |

### Mastery Check

- [ ] Explain why standard attention is memory-bound (N×N to HBM)
- [ ] Implement safe softmax with max-subtract trick
- [ ] Implement online softmax that processes blocks incrementally
- [ ] Implement block-by-block attention with online softmax
- [ ] Explain why block size affects speed but not correctness
- [ ] Calculate memory savings of FlashAttention vs standard
- [ ] Explain the rescaling trick: `O *= exp(old_max - new_max)`